# Equilibrium p-n junction: zero-setup teaching notebook

This notebook is self-contained and runs directly in Google Colab. The symbol
`delta_num` denotes a **numerical smoothing length**, not a Debye length or a
carrier-diffusion length.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp
from scipy.interpolate import interp1d
from scipy.optimize import least_squares
from scipy.sparse import diags

q = 1.602176634e-19
eps0 = 8.8541878128e-12
eps_si = 11.7*eps0
kB = 1.380649e-23

def depletion_edges(Na_cm3, Nd_cm3, Vbi):
    Na, Nd = Na_cm3*1e6, Nd_cm3*1e6
    d = np.sqrt(2*eps_si*Vbi/q*(Na+Nd)/(Na*Nd))
    return d, Nd*d/(Na+Nd), Na*d/(Na+Nd)

def debye_lengths(Na_cm3, Nd_cm3, T):
    Na, Nd = Na_cm3*1e6, Nd_cm3*1e6
    return (np.sqrt(eps_si*kB*T/(q*q*Na)),
            np.sqrt(eps_si*kB*T/(q*q*Nd)))

def smooth_window(x, left, right, delta):
    return 0.5*(np.tanh((x-left)/delta)-np.tanh((x-right)/delta))

def regularised_depletion(Na_cm3, Nd_cm3, T=300., ni_cm3=1e10,
                          buffer_factor=5., points=1201,
                          delta_fraction=0.01):
    Na, Nd = Na_cm3*1e6, Nd_cm3*1e6
    VT = kB*T/q
    Vbi = VT*np.log(Na_cm3*Nd_cm3/ni_cm3**2)
    d, xp, xn = depletion_edges(Na_cm3, Nd_cm3, Vbi)
    LDp, LDn = debye_lengths(Na_cm3, Nd_cm3, T)
    x = np.linspace(-xp-buffer_factor*LDp, xn+buffer_factor*LDn, points)
    delta = delta_fraction*d
    rho = q*(Nd*smooth_window(x, 0., xn, delta)
             - Na*smooth_window(x, -xp, 0., delta))
    u = x/d
    def fun(us, y):
        xs = us*d
        rs = q*(Nd*smooth_window(xs, 0., xn, delta)
                - Na*smooth_window(xs, -xp, 0., delta))
        return np.vstack((y[1], -d*d*rs/eps_si))
    def bc(ya, yb):
        return np.array([ya[0], yb[0]-Vbi])
    guess = np.vstack((Vbi*(u-u[0])/(u[-1]-u[0]),
                       np.full_like(u, Vbi/(u[-1]-u[0]))))
    sol = solve_bvp(fun, bc, u, guess, tol=2e-6, max_nodes=30000)
    if not sol.success:
        raise RuntimeError(sol.message)
    phi = sol.sol(u)[0]
    E = -sol.sol(u)[1]/d
    return dict(x=x, phi=phi, E=E, rho=rho, Vbi=Vbi, d=d, xp=xp, xn=xn,
                delta_num=delta, success=sol.success)


In [ ]:
Na, Nd, T, ni = 1e17, 1e16, 300., 1e10
r = regularised_depletion(Na, Nd, T, ni)
print(f"V_bi = {r['Vbi']:.4f} V")
print(f"x_p = {r['xp']*1e6:.4f} um, x_n = {r['xn']*1e6:.4f} um")
print(f"N_a x_p / (N_d x_n) = {Na*r['xp']/(Nd*r['xn']):.8f}")
print(f"delta_num/d = {r['delta_num']/r['d']:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(r['x']*1e6, r['phi']); ax[0].set(xlabel='x (um)', ylabel='phi (V)')
ax[1].plot(r['x']*1e6, r['E']/1e5); ax[1].set(xlabel='x (um)', ylabel='E (kV/cm)')
for a in ax: a.grid()
plt.tight_layout(); plt.show()

## Interpretive prompts

1. Which side is wider, and how does \(N_a x_p=N_d x_n\) explain it?
2. Change `delta_fraction` from 0.005 to 0.02. Which local features change?
3. Why must `delta_num` not be interpreted as mobile-carrier diffusion?